In [1]:
import pickle
import numpy as np
import pandas as pd
import os
from iblatlas.atlas import BrainRegions
from datetime import datetime
from one.api import ONE

one = ONE()

In [2]:
prefix = '/home/ines/repositories/'

# Load data

In [3]:
import brainwidemap
bwm_df = brainwidemap.bwm_query()
n_sessions = bwm_df['eid'].unique().shape[0]
n_insertions = bwm_df['pid'].unique().shape[0]
print(f'{n_sessions} sessions with {n_insertions} individual neuropixel recordings')
bwm_pid = bwm_df['pid'].unique()

Loading bwm_query results from fixtures/2023_12_bwm_release.csv
459 sessions with 699 individual neuropixel recordings


## LDA axis

In [4]:
data_path = prefix + 'representation_learning_variability/paper-individuality/clustering/'
lda = pd.read_pickle(data_path + 'mouse_LDA_5_bins_cut18-06-2026')
lda = lda.rename(columns={0: 'lda_1'})

lda_eid = lda.loc[lda['session'].isin(list(bwm_df.eid)), 'session']
lda_pid = bwm_df.loc[bwm_df['eid'].isin(lda_eid), 'pid']

print(f'Matching sessions: {len(lda_eid)}')
print(f'Matching insertions: {len(lda_pid)}')

Matching sessions: 244
Matching insertions: 380


In [5]:
# Initialize IBL Brain Atlas and target regions
BRAIN_REGIONS = ['VISa', 'VISam', 'CA1', 'DG', 'LP', 'PO']
regions = BrainRegions()

def get_simplified_area(col_name, filter_sessions=False):
    """Traces an Allen ID/acronym back up to our 6 target regions."""
    raw_acronym = col_name.split('_neuron_')[0]
    allen_ids = regions.acronym2id(raw_acronym)
    beryl_ids = regions.remap(allen_ids, source_map='Allen', target_map='Beryl')
    ancestor = regions.id2acronym(beryl_ids)[0]
    if filter_sessions:
        if ancestor in BRAIN_REGIONS:
            return ancestor
        else:
            return None
    else:
        return ancestor

def get_psth(df, col_name, events, pre=.5, post=1, min_bins_ratio=0.75):
    """
    Extract peri-stimulus time histograms, accounting for data gaps.
    
    Args:
        df: DataFrame with spike data
        col_name: column name for spike counts
        events: trial event times
        pre: time before event (seconds)
        post: time after event (seconds)
        min_bins_ratio: accept windows with at least this fraction of expected bins (0-1).
                       Set to 0.75 to accept windows with ≥75% of expected bins.
    
    Returns:
        all_trials: list of spike count arrays (variable length due to data gaps)
        valid_event_indices: indices of accepted events
        bin_lengths: actual number of bins per trial (for reference)
    """
    all_trials = []
    valid_event_indices = []
    bin_lengths = []
    expected_bins = int((pre + post) * 60)
    min_acceptable_bins = int(expected_bins * min_bins_ratio)
    
    for idx, event in enumerate(events):
        start = event - pre
        end = event + post
        window = df.loc[(df['Bin'] >= start) & (df['Bin'] < end), col_name]
        
        # Accept windows with at least min_bins_ratio of expected bins
        if len(window) >= min_acceptable_bins:
            all_trials.append(window.values)
            valid_event_indices.append(idx)
            bin_lengths.append(len(window))
            
    return all_trials, valid_event_indices, bin_lengths

## Extract firing rates per trial

In [6]:
# PSTH parameters based on 60 Hz sampling rate
MAX_TRIALS_TO_PROCESS = 400 
PSTH_PRE = 0.5   # 500 ms before event
PSTH_POST = 1.0  # 1000 ms after event
MIN_BINS_RATIO = 0.9  # Accept trials with ≥75% of expected bins

# path_dir = '/Users/ineslaranjeira/Google Drive/O meu disco/CCU/PhD Project/paper-individuality/data/neuron_files/'
path_dir = prefix + 'representation_learning_variability/paper-individuality/data/neuron_files/'
save_dir = prefix + 'representation_learning_variability/paper-individuality/data/firing_rates/'

# Create save directory if it doesn't exist
os.makedirs(save_dir, exist_ok=True)

processed_count = 0
skipped_count = 0

for pid in lda_pid:
    # Check if output file already exists
    save_filename = save_dir + "firing_rate_" + str(pid)
    if os.path.exists(save_filename):
        skipped_count += 1
        print(f"Skipping PID (already processed): {pid}")
        continue
    
    filepath = os.path.join(path_dir, f'states_neurons_file_{pid}')
    try:
        summary_records = []
        with open(filepath, 'rb') as f:
            raw_data = pickle.load(f)
            
        state_with_spikes = raw_data.dropna(subset=['Bin']).reset_index(drop=True)
        if state_with_spikes.empty:
            continue
            
        pid_name = state_with_spikes['session_pid'].iloc[0]
        session_name = state_with_spikes['session'].iloc[0]
        
        # --- INFER STIMULUS CONDITION PER TRIAL ---
        def infer_side(row):
            if row['correct'] == 1:
                return "Right" if row['choice'] == 'left' else "Left"
            else:
                return "Left" if row['choice'] == 'right' else "Right"

        state_with_spikes['stim_side'] = state_with_spikes.apply(infer_side, axis=1)
        state_with_spikes['condition'] = state_with_spikes['stim_side'] + "_" + state_with_spikes['contrast'].astype(str)
        
        # Pull chronologically sorted trials, capped at MAX_TRIALS_TO_PROCESS
        all_events = np.array(state_with_spikes['goCueTrigger_times'].unique())
        events_subset = all_events[:MAX_TRIALS_TO_PROCESS]
        
        # Create mapping from event time to trial metadata (trial_id, condition, etc.)
        trial_metadata = state_with_spikes.drop_duplicates(subset=['goCueTrigger_times'])
        trial_metadata_dict = {}
        for _, row in trial_metadata.iterrows():
            trial_metadata_dict[row['goCueTrigger_times']] = {
                'trial_id': row['trial_id'],
                'condition': row['condition']
            }
        
        spike_columns = [col for col in state_with_spikes.columns if '_spike_count' in col]
        
        # Define fixed time axis for all trials (based on expected 60Hz)
        expected_bins = int((PSTH_PRE + PSTH_POST) * 60)
        time_axis = np.linspace(-PSTH_PRE, PSTH_POST, expected_bins)
        time_col_names = [f't_{t:.3f}' for t in time_axis]
        
        # Process each neuron completely independently
        for col in spike_columns:
            area = get_simplified_area(col, filter_sessions=False)
            if not area:  # Skip neurons that don't belong to your target BRAIN_REGIONS
                continue
            
            # Track trials and their actual bin times for alignment
            valid_trials_data = []
            
            for event_idx, event in enumerate(events_subset):
                start = event - PSTH_PRE
                end = event + PSTH_POST
                
                # Get both spike counts AND bin times
                window_mask = (state_with_spikes['Bin'] >= start) & (state_with_spikes['Bin'] < end)
                window_spike_counts = state_with_spikes.loc[window_mask, col].values
                window_bin_times = state_with_spikes.loc[window_mask, 'Bin'].values
                
                # Check if we have enough bins
                min_acceptable_bins = int(expected_bins * MIN_BINS_RATIO)
                if len(window_spike_counts) < min_acceptable_bins:
                    continue
                
                # Create aligned firing rate array with NaNs in correct positions
                firing_rates_aligned = np.full(expected_bins, np.nan, dtype=float)
                sampling_rate = 60
                
                # For each actual bin, find its position in the reference time axis
                for actual_time, spike_count in zip(window_bin_times, window_spike_counts):
                    # Find closest position in the reference time axis
                    time_offset = actual_time - event  # Time relative to event
                    bin_position = int(np.round((time_offset + PSTH_PRE) * sampling_rate))
                    
                    # Only fill if position is within expected range
                    if 0 <= bin_position < expected_bins:
                        firing_rates_aligned[bin_position] = spike_count * sampling_rate
                
                # Count actual bins acquired
                n_bins_actual = np.sum(~np.isnan(firing_rates_aligned))
                
                # Get trial metadata
                trial_info = trial_metadata_dict.get(event, {})
                trial_id = trial_info.get('trial_id', np.nan)
                condition = trial_info.get('condition', np.nan)
                
                valid_trials_data.append({
                    'event_idx': event_idx,
                    'trial_id': trial_id,
                    'event_time': event,
                    'condition': condition,
                    'firing_rates': firing_rates_aligned,
                    'n_bins': n_bins_actual
                })
            
            if len(valid_trials_data) < 10:  # Skip neuron if too few trials
                continue

            # Create one record per trial per neuron
            for trial_data in valid_trials_data:
                record = {
                    'pid': pid_name,
                    'session': session_name,
                    'neuron_id': col,
                    'area': area,
                    'trial_id': trial_data['trial_id'],  # PRESERVE ORIGINAL trial_id
                    'event_time': trial_data['event_time'],
                    'condition': trial_data['condition'],
                    'n_bins': trial_data['n_bins']
                }

                # Add firing rate for each time bin (with NaNs in right places)
                for col_name, fr in zip(time_col_names, trial_data['firing_rates']):
                    record[col_name] = fr

                summary_records.append(record)
        
        # Build DataFrame once after all neurons are processed
        session_data = pd.DataFrame(summary_records)
                            
        # Save per session
        with open(save_filename.replace('.parquet', '.pkl'), 'wb') as f:
            pickle.dump(session_data, f)
        
        processed_count += 1
        print(f"Processed PID: {pid_name}")
        print(f"  Total valid trials: {len(valid_trials_data)}/{len(events_subset)} ({100*len(valid_trials_data)/len(events_subset):.1f}%)")
        
    except Exception as e:
        print(f"Error processing {filepath}: {e}")
        import traceback
        traceback.print_exc()

print(f"\n" + "="*60)
print(f"PROCESSING SUMMARY")
print(f"="*60)
print(f"Newly processed: {processed_count}")
print(f"Already existed (skipped): {skipped_count}")
print(f"Total PIDs: {len(lda_pid)}")


Skipping PID (already processed): 7b05cccc-44f6-4491-a0ea-e38d6e95513d
Skipping PID (already processed): 99993a2b-588e-4c0c-bfec-e3dfb4f61534
Skipping PID (already processed): 298e2a70-9801-45f0-b91c-d6bb9718427e
Skipping PID (already processed): 9dff5593-e781-41a3-a6f9-20d06085e4f8
Skipping PID (already processed): 0909252c-3ad0-413f-96f5-7eff885b50aa
Skipping PID (already processed): e45a00b1-14a0-4f5e-9ea5-9f76d042b11c
Skipping PID (already processed): 117f0d28-3cc0-4837-9e3e-46db5bc3e662
Skipping PID (already processed): 85bdeae3-269b-4e39-bd9b-2b0d95aff2fa
Skipping PID (already processed): 01864e9d-0dbe-41d4-9e3a-0285348ecfc1
Skipping PID (already processed): a9e83d8a-7c90-4152-abad-53a1ad94d73a
Skipping PID (already processed): 0777b1bf-964b-49b7-888b-8a6c9df09c3b
Skipping PID (already processed): 9793c99d-f918-4931-8bba-fdb978bd8e0a
Skipping PID (already processed): 08ed0b3c-9f94-4c1f-8522-3d42a642a6b0
Skipping PID (already processed): a85b9795-f99c-4c1d-a376-8b5ef095ffd7
Skippi